In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Offline (RTX / L4, no internet) Clifford Flow runner. The code comes from a repo snapshot
# dataset (git archive of the branch, see pose3d/USAGE.md "Kaggle"), the libraries from
# syfry5suvzovvakmuj/ga-research-offline-deps, W&B runs offline (gpu_pool/rtx_monitor.py replays the
# WANDB_SYNC log lines, or read them from `kaggle kernels logs`).
#
# Reference run on this data recipe: clifford_flow_cgenn_warp_synth_ema_b64 (kernel
# clifford-cgenn-ema-b64-rtx): real PASCAL3D+ with warp + RenderForCNN synthetic renders from the RAM
# pack syfry5suvzovvakmuj/pascal3d-synth-pack, EMA weights, fixed validation noise, global batch 64
# with lr scaled by sqrt(64/32). 32-sample medoid: 7.88 deg median, 9.82 deg class-mean median.
# BEST_SETUP below is exactly that recipe; EXTRA holds what this run changes.
#
# Attach: syfry5suvzovvakmuj/pascal3d, pascal3d-ram-cache, pascal3d-synth-pack,
# ga-research-offline-deps and ONE repo snapshot dataset.
BEST_SETUP = ['--vector_field', 'clifford', '--condition_head', 'clifford', '--use_warp', '--use_synth',
              '--num_workers', '24', '--ema', '--fixed_val_noise', '--batch_size', '64', '--lr_scaling', 'sqrt']
RUN_NAME = 'clifford_flow_x1_warp_synth_ema_b64'
EXTRA = ['--flow_param', 'x1']          # endpoint prediction, tangent loss (see FlowConfig.flow_param)
SNAPSHOT_MARKER = 'flow_param'          # a string the snapshot's config.py must contain (stale-snapshot guard)
ARGS = BEST_SETUP + EXTRA

In [ ]:
import glob, os, shutil, subprocess, tarfile

# The snapshot dataset is either auto-extracted by Kaggle (a GA-Research/ tree) or still a tar.gz.
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
if markers:
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    repo_root = markers[0][:markers[0].index('/pose3d/pose3d/config.py')]
    shutil.copytree(repo_root, '/kaggle/working/GA-Research')
else:
    tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
print(sorted(os.listdir('/kaggle/working/GA-Research')))
config_src = open('/kaggle/working/GA-Research/pose3d/pose3d/config.py').read()
assert SNAPSHOT_MARKER in config_src, f'stale snapshot: {SNAPSHOT_MARKER!r} is not in config.py'

In [ ]:
# Libraries Kaggle lacks, from the offline wheels (no index, no deps: the image's torch stays).
deps = next(p for p in glob.glob('/kaggle/input/**/ga-research-offline-deps', recursive=True)
            + glob.glob('/kaggle/input/ga-research-offline-deps'))
wheels = sorted(glob.glob(os.path.join(deps, '*.whl')))
print('wheels:', [os.path.basename(w) for w in wheels])
subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)

# torchvision's resnet101 DEFAULT weights, so the pretrained backbone needs no download.
ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
shutil.copy(os.path.join(deps, 'resnet101-cd907fc2.pth'), os.path.join(ckpt_dir, 'resnet101-cd907fc2.pth'))

import importlib, sys
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford', 'opt_einsum']:
    importlib.import_module(mod)
print('python', sys.version.split()[0], '| all offline deps import cleanly')

In [ ]:
# No internet: wandb writes local files only; the trainer also prints WANDB_SYNC lines per epoch.
os.environ['WANDB_MODE'] = 'offline'

In [ ]:
import sys, os
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
PASCAL = next(p for p in ['/kaggle/input/datasets/syfry5suvzovvakmuj/pascal3d', '/kaggle/input/pascal3d']
              if os.path.exists(p))
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={PASCAL}', '--platform=kaggle',
       f'--run_name={RUN_NAME}', *ARGS]
print(' '.join(cmd))
# check=True so a crash marks the Kaggle run as failed instead of passing silently.
subprocess.run(cmd, check=True, env={**os.environ})